二维卷积网络,带批归一化的 VGG 风格 CNN,全连接自编码器，Softmax分类器,双向长短期记忆网络（BiLSTM）,堆叠 LSTM（3 层）

In [10]:
import torch
import torch.nn as nn
class CIFAR(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,10),
        )
    def forward(self,x):
        return self.classifier(self.features(x))
model = CIFAR()
model.eval()
with torch.no_grad():
    out = model(torch.randn(2,3,32,32))
    print('输出形状：', out.shape if isinstance(out, torch.Tensor) else type(out))


输出形状： torch.Size([2, 10])


In [13]:
import torch
import torch.nn as nn
class CNN_BatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(64,128),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(128,10),
        )
    def forward(self,x):
        x = self.block1(x); x = self.block2(x)  
        return self.head(x)

model = CNN_BatchNorm()
model.eval()
with torch.no_grad():
    out = model(torch.randn(2,3,32,32))
    print('输出特征:' ,out.shape if isinstance(out,torch.Tensor) else type(out))

输出特征: torch.Size([2, 10])


In [16]:
import torch
import torch.nn as nn
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784,256),nn.ReLU(),
            nn.Linear(256,64),nn.ReLU(),
            nn.Linear(64,32),
        )
        self.decoder = nn.Sequential(
            nn.Linear(32,64),nn.ReLU(),
            nn.Linear(64,256),nn.ReLU(),
            nn.Linear(256,784),nn.Sigmoid(),
        )

    def forward(self,x):
        z = self.encoder(x)
        return self.decoder(z)

model = Encoder()
model.eval()
with torch.no_grad():
    out = model(torch.randn(4,784))
    print('输出特征:',out.shape if isinstance(out,torch.Tensor) else type(out))

输出特征: torch.Size([4, 784])


In [20]:
import torch
import torch.nn as nn
class SoftmaxClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(30,64),nn.ReLU(),
            nn.Linear(64,32),nn.ReLU(),
            nn.Linear(32,5),nn.Softmax(dim=1),
        )
    def forward(self,x):
        return self.net(x)

model = SoftmaxClassifier()
model.eval()
with torch.no_grad():
    out = model(torch.randn(8,30))
    print('输出特征:',out.shape if isinstance(out,torch.Tensor) else type(out))

输出特征: torch.Size([8, 5])


In [28]:
import torch
import torch.nn as nn
class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=16,hidden_size=32,num_layers=1,batch_first=True,bidirectional=True)
        self.fc = nn.Linear(32 * 2,4)
    def forward(self,x):
        out,(h,c) = self.lstm(x)
        h_cat = torch.cat([h[0],h[1]],dim=1)
        return self.fc(h_cat)
model = BiLSTM()
model.eval()
with torch.no_grad():
    out = model(torch.randn(4,16,16)) 
    print('输出特征:',out.shape if isinstance(out,torch.Tensor) else type(out))


输出特征: torch.Size([4, 4])


In [ ]:
# 例 12：堆叠 LSTM（3 层）
# 类别：循环   |   难度：中级
# 参数量：49,397
# 输入：torch.Size([4, 24, 10])
# 输出：(4, 5)

class Ex12_StackedLSTM(nn.Module):
    """12. Stacked (multi-layer) LSTM with dropout."""
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=10, hidden_size=48, num_layers=3,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(48, 5)
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        return self.fc(h[-1])

model = Ex12_StackedLSTM()
model.eval()
with torch.no_grad():
    out = model(torch.randn(4, 24, 10))
    print('输出形状：', out.shape if isinstance(out, torch.Tensor) else type(out))

输出特征: torch.Size([4, 5])
